# BirdCLEF+ 2026 — Phase 3 Inference (Local)

Loads:
- **Perch v2 ONNX** from `data/perch/perch_v2_no_dft.onnx`
- **Trained head** from `checkpoints/model_v5.pt`

Pipeline per soundscape file:

```
60s audio → 12 × 5s chunks → Perch ONNX → 12 × 1536-d embeddings
                                                 ↓ PerchHead
                                          12 × 234 logits
                                                 ↓ Gaussian smooth (logit space)
                                                 ↓ sigmoid
                                                 ↓ submission.csv row
```

Locally, `test_soundscapes/` is empty (Kaggle reveals the real test only during scored
submission), so we fall back to 5 files from `train_soundscapes/` to verify the pipeline.


## 1. Setup


In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torch.nn as nn
import onnxruntime as ort
from scipy.ndimage import convolve1d
from tqdm.auto import tqdm

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "birdclef-2026"
PERCH_PATH   = PROJECT_ROOT / "data" / "perch" / "perch_v2_no_dft.onnx"
CKPT_PATH    = PROJECT_ROOT / "checkpoints" / "model_v5.pt"
OUT_PATH     = PROJECT_ROOT / "submission.csv"

assert PERCH_PATH.exists(), f"Missing {PERCH_PATH}"
assert CKPT_PATH.exists(),  f"Missing {CKPT_PATH} — run Phase 3 training first"


## 2. Load Perch + the head

Perch ONNX needs ~400MB of disk read on first load but ~1 sec to warm up after.
The head is tiny (a few MB) — loads instantly.


In [ ]:
# Perch
sess = ort.InferenceSession(str(PERCH_PATH), providers=["CPUExecutionProvider"])
INPUT_NAME    = sess.get_inputs()[0].name
EMBED_OUT_IDX = next(i for i, o in enumerate(sess.get_outputs()) if o.name == "embedding")

# Head
ckpt          = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
species       = ckpt["species"]
label_to_idx  = ckpt["label_to_idx"]
NUM_CLASSES   = ckpt["num_classes"]
EMBED_DIM     = ckpt["embed_dim"]
head_config   = ckpt["head_config"]


class PerchHead(nn.Module):
    def __init__(self, embed_dim=1536, hidden_dim=512, num_classes=234, dropout=0.3):
        super().__init__()
        self.norm  = nn.LayerNorm(embed_dim)
        self.drop1 = nn.Dropout(0.2)
        self.fc1   = nn.Linear(embed_dim, hidden_dim)
        self.act   = nn.ReLU(inplace=True)
        self.drop2 = nn.Dropout(dropout)
        self.fc2   = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        x = self.norm(x)
        x = self.drop1(x)
        x = self.act(self.fc1(x))
        x = self.drop2(x)
        return self.fc2(x)


head = PerchHead(EMBED_DIM, **head_config, num_classes=NUM_CLASSES).to(DEVICE)
head.load_state_dict(ckpt["state_dict"])
head.eval()
print(f"Head loaded: {NUM_CLASSES} classes, hidden={head_config['hidden_dim']}")


## 3. Audio + windowing helpers (same as before)


In [ ]:
SR          = 32000
CLIP_SEC    = 5
N_SAMPLES   = SR * CLIP_SEC
WINDOW_SEC  = CLIP_SEC
N_WINDOWS   = 60 // WINDOW_SEC  # 12

def load_audio(path):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav.astype(np.float32)


def file_to_chunks(path):
    wav    = load_audio(path)
    target = N_WINDOWS * N_SAMPLES
    if len(wav) < target:
        wav = np.pad(wav, (0, target - len(wav)))
    else:
        wav = wav[:target]
    return wav.reshape(N_WINDOWS, N_SAMPLES).astype(np.float32)


GAUSSIAN_KERNEL = np.array([0.1, 0.2, 0.4, 0.2, 0.1])

def smooth_windows(logits):
    return convolve1d(logits, GAUSSIAN_KERNEL, axis=0, mode="nearest")


## 4. Inference function: file → 12 × 234 probabilities


In [ ]:
@torch.no_grad()
def predict_file(path):
    chunks = file_to_chunks(path)                                # (12, 160000)
    # Perch on all 12 in one batch
    embeddings = sess.run(None, {INPUT_NAME: chunks})[EMBED_OUT_IDX]  # (12, 1536)
    # Head
    embeddings = torch.from_numpy(embeddings).to(DEVICE)
    logits     = head(embeddings).cpu().numpy()                  # (12, NUM_CLASSES)
    logits     = smooth_windows(logits)
    return 1.0 / (1.0 + np.exp(-logits))


# Smoke test
TEST_DIR  = DATA_DIR / "test_soundscapes"
TRAIN_DIR = DATA_DIR / "train_soundscapes"
test_files = sorted(TEST_DIR.glob("*.ogg")) if TEST_DIR.is_dir() else []
if not test_files:
    test_files = sorted(TRAIN_DIR.glob("*.ogg"))[:5]
    print(f"[fallback] using {len(test_files)} files from train_soundscapes/")
else:
    print(f"Found {len(test_files)} test files")

probs0 = predict_file(test_files[0])
print(f"output shape: {probs0.shape}  range: [{probs0.min():.3f}, {probs0.max():.3f}]")


## 5. Run inference on all files


In [ ]:
all_rows, all_probs = [], []
t0 = time.time()
for f in tqdm(test_files, desc="infer"):
    basename = f.stem
    probs    = predict_file(f)
    end_secs = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
    for k in range(N_WINDOWS):
        all_rows.append(f"{basename}_{end_secs[k]}")
        all_probs.append(probs[k])
all_probs = np.stack(all_probs)
print(f"{len(all_rows)} rows in {time.time()-t0:.1f}s")


## 6. Write submission.csv with correct column order


In [ ]:
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
all_species_in_order = [c for c in sample_sub.columns if c != "row_id"]

pred_df = pd.DataFrame(all_probs, columns=species)
pred_df.insert(0, "row_id", all_rows)
sub = pred_df.set_index("row_id").reindex(columns=all_species_in_order)
# Species the model doesn't know about (in taxonomy but not in our training set) → uniform prior
sub = sub.fillna(1.0 / len(all_species_in_order)).clip(0.0, 1.0).reset_index()

assert list(sub.columns) == list(sample_sub.columns), "Column order mismatch!"
assert sub["row_id"].is_unique
assert not sub.isna().any().any()

sub.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}  shape={sub.shape}  size={OUT_PATH.stat().st_size/1024:.1f} KB")
sub.head(3)


### What's next

When this looks good (probabilities span a reasonable range, no errors), I'll push
the corresponding Kaggle kernel with the same logic but Kaggle paths. Then you
submit from the CLI like before.
